# 12 — Supervised volatility forecasts: HAR null, implied variance and HAR-X

**Question.** Does the market state at $t$ forecast the **future realized variance of the Deribit index** beyond volatility persistence, the deterministic clock and the market's own volatility price? Design agreed in `discussions/001`–`004`.

**Target.** $y_{t,h}=\log(RV_{t,h}+\epsilon_{asset,h})$, $RV_{t,h}=\sum_{j=1}^{m} r_{t+j}^2$ of five-minute index log returns, $h\in\{30\text{m},60\text{m}\}$ (co-primary). Targets crossing the 08:00 UTC expiry are dropped. Features are measured at the close of bar $t$; targets start at $r_{t+1}$.

**Terminal target (provisional, descriptive).** $\sum_{j>t} w_j^2(\Delta S_j/S_t)^2$, the martingale variance of the 30-minute settlement average on the five-minute grid (004 §3).

**Nested models.** $H$ = HAR + clock is never penalized; the added block $X$ is Ridge-shrunk after residualization on $H$ (Frisch–Waugh–Lovell), so every model nests the HAR null exactly.

| Model | Adds |
|---|---|
| M0 | persistence: $\log RV^{back}_{30m}+\log(m/6)$ |
| M1 | HAR + New York clock: $\ell^{5m},\ell^{30m},\ell^{1h},\ell^{4h}$, forward and backward clock variance, weekend |
| M1c | M1 with a training-only affine recalibration (out-of-fold) — the Module 13 null |
| M2a | $\log \widetilde W^{imp}_{t,t+h}$: clock-allocated implied integrated variance |
| M2b | M2a + ATM IV changes (15m, 30m) |
| M3 | M2b + returns, flow, L2, basis/OI, $\lvert D\rvert$, $D$, $\Delta\lvert D\rvert$, perp range / Parkinson / Garman–Klass |
| M4 | M3 + option shape, dynamics and liquidity |

**Scores.** QLIKE on the variance level (Duan smearing) is primary; squared log error secondary; Gaussian / Student-t log scores, $R^2$, change-form $R^2$ and Mincer–Zarnowitz are diagnostics. Comparisons are session-paired; a non-overlapping-origin view is mandatory.

**Status.** Sessions through 2026-10-07 are a **development / screening** sample: they check the specification, choose $\lambda$ and freeze the models. Nothing here is confirmatory. The binding test is the hierarchy below on the first **30 completed sessions after 2026-10-07**, scored with the frozen specification (`NB12_MODE=confirmation`):

1. M2a vs M1 — does horizon-matched implied variance add physical-RV information?
2. M2b vs M2a — do recent IV dynamics add?
3. M3 vs M2b — do futures / L2 / flow / basis / D add?
4. M4 vs M3 — does option-surface information add?

"IV adds nothing" would **not** mean options are mispriced (004 §10); no economic claim is made here.

In [1]:
from pathlib import Path
import os, sys, time, json, subprocess, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 260)

load_dotenv()
DATA_ROOT = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve()
DB_PATH = DATA_ROOT / "database" / "research.duckdb"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# A kernel that imported older versions keeps them cached: purge before importing.
for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling import full_information as fi
from data_engineering.modeling import vol_targets as vt
from data_engineering.modeling import vol_har as vh
from data_engineering.modeling.clock_baseline import CLOCK_BASELINE_VERSION
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import blocked_folds

if vt.VOL_TARGETS_VERSION < "0.1" or vh.VOL_HAR_VERSION < "0.1":
    raise RuntimeError("Old Module 12 code: git pull and restart the kernel.")

MODE = os.getenv("NB12_MODE", "development")          # development | confirmation
ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
DEV_START, SCREEN_TRAIN_END, DEV_END = "2026-09-22", "2026-09-28", "2026-10-07"
MIN_FEATURE_COVERAGE = float(os.getenv("NB12_MIN_FEATURE_COVERAGE", "0.45"))
USE_EN = os.getenv("NB12_EN", "1") == "1"
EN_GRID = (1e-3, 1e-2, 1e-1) if USE_EN else ()
N_BOOT = int(os.getenv("NB12_BOOT", "2000"))
FREEZE = os.getenv("NB12_FREEZE", "1") == "1"
REFREEZE = os.getenv("NB12_REFREEZE", "0") == "1"
SPEC_PATH = PROJECT_ROOT / "notebooks" / "12_frozen_spec.json"
CONFIRM_MIN_SESSIONS = 30
HORIZONS = list(vt.HORIZONS)

print(
    f"mode {MODE} | vol_targets {vt.VOL_TARGETS_VERSION} | vol_har {vh.VOL_HAR_VERSION} | "
    f"clock {CLOCK_BASELINE_VERSION} | development {DEV_START} -> {DEV_END} "
    f"(screening fit through {SCREEN_TRAIN_END}) | lambda grid {vh.LAMBDA_GRID} | EN {EN_GRID}"
)

mode development | vol_targets 0.1 | vol_har 0.1 | clock 0.2 | development 2026-09-22 -> 2026-10-07 (screening fit through 2026-09-28) | lambda grid (0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0) | EN (0.001, 0.01, 0.1)


## 1. Load the state table

In development mode only sessions up to 2026-10-07 are read: confirmation data is never touched before the specification is frozen.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])

required = {
    "state_id", "state_time", "underlying", "return_5m", "futures_index_price", "is_weekend",
    "in_settlement_window", "vwap", "volume_usd", "futures_tradable_mid_price", "trade_count",
    "perp_integrity_ok", "known_exchange_event", "integrity_flag_trades_outside_bbo",
    "microstructure_features_ready", "instability_features_ready", "option_snapshot_ready",
    "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
    "option_iv_atm_forward",
}
optional = {"open", "high", "low", "close"}            # perp trade OHLC: range regressors only
missing_required = sorted(required - cols)
if missing_required:
    raise RuntimeError(f"time_series_feature_state lacks {missing_required}")
want = sorted((required | optional | {s.column for s in fi.FEATURE_SPECS}) & cols)
if optional - cols:
    print(f"Perp OHLC columns missing ({sorted(optional - cols)}): Parkinson / Garman-Klass regressors will be empty.")

DEV_LAST_BAR = pd.Timestamp(f"{DEV_END} 08:00", tz="UTC")     # closes the last development expiry
dev_filter = "" if MODE == "confirmation" else (
    f"WHERE state_time <= TIMESTAMPTZ '{DEV_LAST_BAR.isoformat()}'"
)
df = con.execute(
    f"SELECT {', '.join(want)} FROM time_series_feature_state {dev_filter} ORDER BY underlying, state_time"
).df()
con.close()

df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
df = df[df["underlying"].isin(ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
if not (df["session_date"].to_numpy() == vt.expiry_time(df["state_time"]).sub(pd.Timedelta(hours=8)).dt.strftime("%Y-%m-%d").to_numpy()).all():
    raise RuntimeError("Session keys of regime_inputs and vol_targets disagree.")

if MODE != "confirmation":
    # The bar closing at DEV_END 08:00 carries the last return of the final development
    # expiry; it is kept only as a target increment (its session is never an origin).
    keep = ((df["session_date"] >= DEV_START) & (df["session_date"] <= DEV_END)) | (df["state_time"] == DEV_LAST_BAR)
    df = df[keep].reset_index(drop=True)
    assert df["state_time"].max() <= DEV_LAST_BAR, "confirmation data leaked into development"

DEV_SESSIONS = sorted(s for s in df["session_date"].unique() if DEV_START <= s <= DEV_END)
SCREEN_TRAIN = {s for s in DEV_SESSIONS if s <= SCREEN_TRAIN_END}
SCREEN_EVAL = {s for s in DEV_SESSIONS if s > SCREEN_TRAIN_END}
print(
    f"Rows {len(df):,} | development sessions {len(DEV_SESSIONS)} "
    f"(screening fit {len(SCREEN_TRAIN)}, screening evaluation {len(SCREEN_EVAL)})"
)
if len(SCREEN_TRAIN) < 5 or len(SCREEN_EVAL) < 3:
    raise RuntimeError("Module 12 needs the Sep 22-28 sessions and the later development sessions.")

Rows 27,648 | development sessions 16 (screening fit 7, screening evaluation 9)


## 2. Inputs per asset

* **Panel** (`vol_targets.build_panel`): forward / backward realized variance, terminal target, perp range estimators, ATM IV (decimal, quote-backed rows outside the settlement window).
* **D features**: session-VWAP displacement exactly as in Notebooks 08–11, re-indexed to the origin row ($D_t$, not $D_{t-1}$): $\lvert D_t\rvert$, $D_t$, $\Delta\lvert D_t\rvert$. $D$ uses the clock of the window it is built for (screening fit or full development); it is not refitted inside CV folds.
* **Engineered state** (`full_information.FEATURE_SPECS`), measured at the origin row. Eligibility (≥ 45% training coverage, ≥ 3 distinct values) is decided on training rows only.

In [3]:
def asset_frame(asset, frame=None):
    frame = df if frame is None else frame
    return frame[frame["underlying"] == asset].sort_values("state_time").reset_index(drop=True)


def d_features(g, nuis):
    """|D_t|, D_t and d|D_t| at the origin row, with the nuisance's clock."""
    cbs = nuis.clock_set()
    clock_frame = g[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]]
    f = cbs.transform(clock_frame)["return_5m__clock_vol_factor"].fillna(1.0).to_numpy()
    d = build_design(g, f)
    D = np.full(len(g), np.nan)
    src = d.frame["source_row"].to_numpy(int)
    D[src - 1] = d.frame["D"].to_numpy(float)          # frame D at row s is D_{s-1}
    t = pd.to_datetime(g["state_time"], utc=True)
    contig = np.r_[False, (t.diff().iloc[1:] == vt.BAR).to_numpy()]
    absD = np.abs(D)
    dabs = np.where(contig, absD - np.r_[np.nan, absD[:-1]], np.nan)
    return pd.DataFrame({"absD": absD, "D": D, "d_absD": dabs})


def raw_features(g, nuis):
    raw, used, _ = fi.raw_feature_frame(g)
    return pd.concat([raw.reset_index(drop=True), d_features(g, nuis)], axis=1), used


def eligibility(mat, used_specs, train_rows, h):
    X = mat.X[h]
    cov = X.loc[train_rows].notna().mean()
    nun = X.loc[train_rows].nunique(dropna=True)
    avail = {c for c in X.columns if cov.get(c, 0.0) >= MIN_FEATURE_COVERAGE and nun.get(c, 0) >= 3}
    specs_ok = [s for s in used_specs if s.name in avail]
    return avail, fi.block_map(specs_ok), cov


def prepare(asset, train_sessions, frame=None):
    """Panel, nuisance, raw features and Material for one training window."""
    g = asset_frame(asset, frame)
    panel = vt.build_panel(g)
    nuis = vh.fit_nuisance(g, panel, train_sessions)
    raw, used = raw_features(g, nuis)
    mat = vh.materialize(panel, nuis, raw)
    return dict(g=g, panel=panel, nuis=nuis, raw=raw, used=used, mat=mat)


INPUTS = {a: prepare(a, SCREEN_TRAIN) for a in ASSETS}

# Causality: scrambling everything after a cut must leave all origin-row inputs unchanged.
for a, b in INPUTS.items():
    vt.alignment_selftest(b["g"])
print("Alignment self-test passed for", ", ".join(INPUTS))

cov_rows = []
for a, b in INPUTS.items():
    P, m = b["panel"], b["mat"]
    for h in HORIZONS:
        dev = np.isin(P.session_date, DEV_SESSIONS)
        cov_rows.append({
            "underlying": a, "horizon": h, "origins": int(dev.sum()),
            "target_share": float(np.isfinite(m.y[h][dev]).mean()),
            "W_imp_share": float(np.isfinite(m.X[h]["log_W_imp_alloc"].to_numpy()[dev]).mean()),
            "D_share": float(np.isfinite(b["raw"]["D"].to_numpy()[dev]).mean()),
            "log10_eps": np.log10(b["nuis"].eps_target[h]),
            "iv_divisor": P.iv_divisor,
            "median_iv": float(np.nanmedian(P.iv)),
        })
print("Target and input coverage (development sessions):")
display(pd.DataFrame(cov_rows).round(4))

Alignment self-test passed for AVAX, BTC, ETH, HYPE, SOL, XRP
Target and input coverage (development sessions):


,underlying,horizon,origins,target_share,W_imp_share,D_share,log10_eps,iv_divisor,median_iv
0,AVAX,30m,4608,0.8845,0.9125,0.7717,-5.4438,100.0,0.7640
1,AVAX,60m,4608,0.8058,0.9084,0.7717,-4.9108,100.0,0.7640
2,BTC,30m,4608,0.8845,0.9143,0.7728,-6.8011,100.0,0.2504
3,BTC,60m,4608,0.8058,0.9123,0.7728,-6.2319,100.0,0.2504
4,ETH,30m,4608,0.8845,0.9201,0.7728,-6.4400,100.0,0.3172
5,ETH,60m,4608,0.8058,0.9186,0.7728,-5.9360,100.0,0.3172
6,HYPE,30m,4608,0.8845,0.9067,0.7728,-5.9636,100.0,0.6158
7,HYPE,60m,4608,0.8058,0.9056,0.7728,-5.4766,100.0,0.6158
8,SOL,30m,4608,0.8845,0.8635,0.7726,-5.9160,100.0,0.5247
9,SOL,60m,4608,0.8058,0.8631,0.7726,-5.3490,100.0,0.5247


## 3. Target diagnostics

Overlap makes consecutive 30m / 60m targets strongly autocorrelated; inference is session-blocked throughout. The terminal target is compared with its log-return analogue (004 §3) and with the clock-allocated implied variance of the settlement average.

In [4]:
diag = []
for a, b in INPUTS.items():
    m, P = b["mat"], b["panel"]
    for h in HORIZONS:
        y = pd.Series(m.y[h])
        lag = vt.HORIZONS[h]
        diag.append({
            "underlying": a, "horizon": h, "n": int(y.notna().sum()),
            "mean": y.mean(), "sd": y.std(), "skew": y.skew(),
            "ac_lag1": y.autocorr(1), f"ac_lag_m": y.autocorr(lag),
            "corr_y_l30m": pd.Series(m.y[h]).corr(m.H[h]["l30m"]),
        })
print("Log-RV targets:")
display(pd.DataFrame(diag).round(3))

term = []
for a, b in INPUTS.items():
    s = b["mat"].settle
    ok = np.isfinite(s.rv_settle) & (s.rv_settle > 0)
    w = ok & np.isfinite(s.W_imp_settle) & (s.W_imp_settle > 0)
    term.append({
        "underlying": a,
        "finite_share": float(ok.mean()),
        "median_level_over_log": float(np.nanmedian(s.rv_settle[ok] / s.rv_settle_log[ok])),
        "p99_abs_log_diff": float(np.nanquantile(np.abs(np.log(s.rv_settle[ok] / s.rv_settle_log[ok])), 0.99)),
        "corr_log_rv_settle_log_W": float(np.corrcoef(np.log(s.rv_settle[w]), np.log(s.W_imp_settle[w]))[0, 1]) if w.sum() > 10 else np.nan,
        "median_log_rv_over_W": float(np.nanmedian(np.log(s.rv_settle[w] / s.W_imp_settle[w]))) if w.any() else np.nan,
    })
print("Provisional terminal target (descriptive only):")
display(pd.DataFrame(term).round(4))

Log-RV targets:


,underlying,horizon,n,mean,sd,skew,ac_lag1,ac_lag_m,corr_y_l30m
0,AVAX,30m,4076,-10.289,0.944,0.267,0.915,0.470,0.471
1,AVAX,60m,3713,-9.405,0.789,0.277,0.969,0.551,0.502
2,BTC,30m,4076,-12.632,1.178,0.140,0.938,0.609,0.606
3,BTC,60m,3713,-11.781,1.034,0.210,0.978,0.672,0.625
4,ETH,30m,4076,-12.195,1.117,0.361,0.937,0.602,0.594
5,ETH,60m,3713,-11.354,0.994,0.397,0.977,0.638,0.601
6,HYPE,30m,4076,-11.056,0.940,0.097,0.918,0.504,0.502
7,HYPE,60m,3713,-10.226,0.796,0.066,0.971,0.566,0.527
8,SOL,30m,4076,-11.362,0.951,0.462,0.919,0.502,0.488
9,SOL,60m,3713,-10.471,0.791,0.557,0.971,0.576,0.518


Provisional terminal target (descriptive only):


,underlying,finite_share,median_level_over_log,p99_abs_log_diff,corr_log_rv_settle_log_W,median_log_rv_over_W
0,AVAX,0.3898,0.9995,0.0630,0.8318,0.1725
1,BTC,0.3898,1.0000,0.0267,0.8584,0.3298
2,ETH,0.3898,1.0002,0.0295,0.8503,0.1597
3,HYPE,0.3898,1.0008,0.0501,0.8847,0.0388
4,SOL,0.3898,1.0001,0.0476,0.8114,-0.1774
5,XRP,0.3898,0.9985,0.0466,0.8082,-0.0791


## 4. Development screening

Fit on 2026-09-22 → 09-28, evaluate on 09-29 → 10-07. Inside the fit window, $\lambda$ is chosen by blocked expanding-window CV on QLIKE with every nuisance quantity refitted per fold, using a **one-standard-error rule toward the larger penalty** (004 §7). M1c is recalibrated on out-of-fold M1 forecasts. OLS on M4 is a diagnostic; Elastic Net on M4 is the sparse challenger.

These nine evaluation sessions were inspected in Modules 08–11: the readout below is screening, not a test.

In [5]:
def fit_window(asset, b, train_sessions, h):
    """CV, penalty choice and final fits for one asset / horizon / window."""
    g, P, mat = b["g"], b["panel"], b["mat"]
    tr = np.isin(mat.sessions, sorted(train_sessions))
    avail, blocks, cov = eligibility(mat, b["used"], tr, h)
    specs = vh.model_specs(blocks, avail)
    folds = blocked_folds(sorted(train_sessions), n_folds=3, min_train_share=0.45)

    def build(trs):
        return vh.materialize(P, vh.fit_nuisance(g, P, trs), b["raw"])

    cv, extra = vh.cross_validate(build, folds, h, specs, en_grid=EN_GRID)
    lam = {}
    for m in ("M2a", "M2b", "M3", "M4"):
        lam[m] = vh.one_se_choice(cv, m)[0] if specs[m] else 0.0
        if not np.isfinite(lam[m]):
            lam[m] = max(vh.LAMBDA_GRID)
    en_alpha = vh.one_se_choice(cv, "M4", "elasticnet")[0] if EN_GRID else np.nan

    fits = {"M0": vh.fit_vol(mat, h, "M0", [], tr)}
    fits["M1"] = vh.fit_vol(mat, h, "M1", [], tr)
    fits["M1c"] = vh.calibrate_m1(fits["M1"], extra["oof_m1"])
    for m in ("M2a", "M2b", "M3", "M4"):
        fits[m] = vh.fit_vol(mat, h, m, specs[m], tr, lam=lam[m])
    fits["M4_ols"] = vh.fit_vol(mat, h, "M4_ols", specs["M4"], tr, lam=0.0)
    if EN_GRID and np.isfinite(en_alpha):
        fits["M4_en"] = vh.fit_vol(mat, h, "M4_en", specs["M4"], tr, kind="elasticnet", lam=en_alpha)
    return dict(specs=specs, avail=avail, blocks=blocks, cv=cv, lam=lam, en_alpha=en_alpha, fits=fits, coverage=cov)


SCREEN = {}
t_all = time.perf_counter()
for a, b in INPUTS.items():
    t0 = time.perf_counter()
    SCREEN[a] = {h: fit_window(a, b, SCREEN_TRAIN, h) for h in HORIZONS}
    print(f"{a}: screening fits in {time.perf_counter() - t0:.0f}s")
print(f"total {time.perf_counter() - t_all:.0f}s")

lam_rows = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        lam_rows.append({"underlying": a, "horizon": h, **{f"lam_{m}": v for m, v in r["lam"].items()},
                         "en_alpha_M4": r["en_alpha"], **{f"n_{m}": len(r["specs"][m]) for m in ("M2b", "M3", "M4")}})
print("Selected penalties (one-SE toward larger) and model sizes:")
display(pd.DataFrame(lam_rows))

AVAX: screening fits in 43s
BTC: screening fits in 36s
ETH: screening fits in 39s
HYPE: screening fits in 37s
SOL: screening fits in 38s
XRP: screening fits in 41s
total 234s
Selected penalties (one-SE toward larger) and model sizes:


,underlying,horizon,lam_M2a,lam_M2b,lam_M3,lam_M4,en_alpha_M4,n_M2b,n_M3,n_M4
0,AVAX,30m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,65
1,AVAX,60m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,65
2,BTC,30m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,63
3,BTC,60m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,63
4,ETH,30m,10000.0,10000.0,10.0,10000.0,0.1,3,36,65
5,ETH,60m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,65
6,HYPE,30m,10000.0,10000.0,10000.0,10000.0,0.1,3,34,63
7,HYPE,60m,10000.0,10000.0,10000.0,10000.0,0.1,3,34,63
8,SOL,30m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,65
9,SOL,60m,10000.0,10000.0,10000.0,10000.0,0.1,3,36,65


In [6]:
SCORES, metrics = {}, []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    SCORES[a] = {}
    for h, r in by_h.items():
        SCORES[a][h] = {m: vh.row_scores(fv, mat, h) for m, fv in r["fits"].items()}
        y = mat.y[h]
        dy = y - mat.delta_base[h]
        ref = {k: SCORES[a][h][k]["yhat"].to_numpy() for k in ("M1", "M2b") if k in SCORES[a][h]}
        for m, sc in SCORES[a][h].items():
            yhat = sc["yhat"].to_numpy()
            rows = te & np.isfinite(sc["qlike"].to_numpy())
            rec = {
                "underlying": a, "horizon": h, "model": m, "n": int(rows.sum()),
                "qlike": float(sc["qlike"][rows].mean()), "log_rmse": float(np.sqrt(sc["sqerr"][rows].mean())),
                "ls_gauss": float(sc["ls_gauss"][rows].mean()), "ls_t": float(sc["ls_t"][rows].mean()),
                "r2": vh.r2(y[rows], yhat[rows]), "r2_change_form": vh.r2(dy[rows], (yhat - mat.delta_base[h])[rows]),
            }
            for k, yr in ref.items():
                both = rows & np.isfinite(yr)
                rec[f"dR2_vs_{k}"] = vh.r2(y[both], yhat[both]) - vh.r2(y[both], yr[both])
            rec.update(vh.mz(y[rows], yhat[rows], mat.sessions[rows], n_boot=500))
            metrics.append(rec)
metrics = pd.DataFrame(metrics)
print("Screening metrics on 2026-09-29 -> 10-07 (each model on its own complete rows):")
display(metrics.round(4))

Screening metrics on 2026-09-29 -> 10-07 (each model on its own complete rows):


,underlying,horizon,model,n,qlike,log_rmse,ls_gauss,ls_t,r2,r2_change_form,dR2_vs_M1,dR2_vs_M2b,mz_a,mz_b,mz_b_lo,mz_b_hi
0,AVAX,30m,M0,2093,0.4557,0.9036,-1.3385,-1.3337,0.1272,-0.0001,-0.1699,-0.2003,-4.5717,0.5646,0.3877,0.6575
1,AVAX,30m,M1,1257,0.3764,0.8104,-1.2152,-1.2283,0.3036,0.1961,0.0000,-0.0389,6.5075,1.6575,1.2999,1.8643
2,AVAX,30m,M1c,1257,0.4311,0.8806,-1.3074,-1.3363,0.1778,0.0509,-0.1258,-0.1562,12.7694,2.2880,1.7944,2.5734
3,AVAX,30m,M2a,1131,0.3656,0.8008,-1.2079,-1.2075,0.3302,0.2356,0.0321,-0.0071,5.2531,1.5308,1.1474,1.7416
4,AVAX,30m,M2b,1049,0.3741,0.8042,-1.2135,-1.2125,0.3402,0.2329,0.0389,0.0000,5.3146,1.5334,1.1664,1.7303
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,XRP,60m,M2b,775,0.3510,0.8269,-1.5561,-1.5463,0.1891,0.0583,0.0187,0.0000,-0.0624,1.0344,0.6583,1.3105
104,XRP,60m,M3,532,0.3267,0.7772,-1.3645,-1.3592,0.2433,0.0432,0.1286,0.1134,-1.4533,0.8903,0.4453,1.1329
105,XRP,60m,M4,129,0.2705,0.7498,-1.7233,-1.6613,0.1919,0.2916,-0.0476,-0.0353,-2.4600,0.7584,-1.1605,1.2258
106,XRP,60m,M4_ols,129,0.3257,0.9086,-5.0235,-3.8129,-0.1867,-0.0403,-0.4262,-0.4139,-2.1511,0.8330,0.6253,1.1084


### Paired comparisons

Positive = the first model is better. `qlike_*` and `sqerr_*` are session sums of loss reductions; `ls_t_*` of Student-t log-score gains. Each pair is scored on the rows both models can score. The non-overlapping view keeps only session-aligned origins every $h$.

In [7]:
PAIRS = vh.HIERARCHY + [("M1c", "M1"), ("M4_en", "M4"), ("M4_ols", "M4")]
cmp_rows = []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    for h in by_h:
        nov = vh.non_overlapping(mat.tte_min, h)
        for x, y_ in PAIRS:
            if x not in SCORES[a][h] or y_ not in SCORES[a][h]:
                continue
            for view, rows in (("all", te), ("non_overlapping", te & nov)):
                c = vh.compare(SCORES[a][h][x], SCORES[a][h][y_], mat.sessions, rows, n_boot=N_BOOT)
                cmp_rows.append({"underlying": a, "horizon": h, "model": x, "vs": y_, "view": view, **c})
comparisons = pd.DataFrame(cmp_rows)
show = ["underlying", "horizon", "model", "vs", "view", "n_rows", "qlike_n_sessions", "qlike_mean_session_diff",
        "qlike_boot_lo", "qlike_boot_hi", "qlike_sessions_won", "sqerr_mean_session_diff", "ls_t_mean_session_diff"]
display(comparisons[show].round(4))

,underlying,horizon,model,vs,view,n_rows,qlike_n_sessions,qlike_mean_session_diff,qlike_boot_lo,qlike_boot_hi,qlike_sessions_won,sqerr_mean_session_diff,ls_t_mean_session_diff
0,AVAX,30m,M1,M0,all,1257,9,11.5983,0.7544,22.6176,0.7778,22.3813,14.6022
1,AVAX,30m,M1,M0,non_overlapping,216,9,2.0194,-0.4117,4.7819,0.6667,4.7896,2.4016
2,AVAX,30m,M2a,M1,all,1131,9,2.4130,-0.6951,5.1307,0.7778,3.8606,4.1711
3,AVAX,30m,M2a,M1,non_overlapping,195,9,0.4371,-0.0944,0.9010,0.7778,0.6772,0.8502
4,AVAX,30m,M2b,M2a,all,1049,9,0.0368,-0.9289,0.8245,0.7778,0.8075,0.7646
...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,XRP,60m,M1c,M1,non_overlapping,102,9,-0.3743,-0.7377,-0.0277,0.3333,0.1015,1.6982
188,XRP,60m,M4_en,M4,all,129,6,0.0816,-0.4364,0.5722,0.6667,-0.3104,-8.3013
189,XRP,60m,M4_en,M4,non_overlapping,11,5,-0.0283,-0.0934,0.0263,0.4000,-0.1227,-1.3665
190,XRP,60m,M4_ols,M4,all,129,6,-1.1855,-5.1825,1.2506,0.6667,-5.6629,-46.2583


### Population retained by each model (004 §11)

Option-based models score only quote-backed rows outside the settlement window. This table shows what population remains.

In [8]:
def tte_bucket(tte):
    return pd.cut(tte / 60.0, [-1, 4, 8, 16, 24.5], labels=["0-4h", "4-8h", "8-16h", "16-24h"])

sel = []
for a, by_h in SCREEN.items():
    b = INPUTS[a]
    mat, raw = b["mat"], b["raw"]
    dev = np.isin(mat.sessions, DEV_SESSIONS)
    spread = raw["log_option_spread_med"].to_numpy() if "log_option_spread_med" in raw else np.full(len(raw), np.nan)
    for h, r in by_h.items():
        elig = dev & np.isfinite(mat.y[h]) & np.isfinite(mat.H[h][vh.H_NAMES].to_numpy(float)).all(axis=1)
        for m in ("M1", "M2a", "M2b", "M3", "M4"):
            keep = elig & vh.complete(mat, h, r["specs"][m])
            rec = {"underlying": a, "horizon": h, "model": m, "eligible": int(elig.sum()), "retained": int(keep.sum()),
                   "retained_share": float(keep.sum() / max(elig.sum(), 1)),
                   "weekend_share_retained": float(INPUTS[a]["panel"].weekend[keep].mean()) if keep.any() else np.nan,
                   "min_session_share": float(pd.Series(keep[elig]).groupby(mat.sessions[elig]).mean().min()),
                   "median_y_retained": float(np.nanmedian(mat.y[h][keep])) if keep.any() else np.nan,
                   "median_y_excluded": float(np.nanmedian(mat.y[h][elig & ~keep])) if (elig & ~keep).any() else np.nan,
                   "median_log_opt_spread_retained": float(np.nanmedian(spread[keep])) if keep.any() else np.nan,
                   "median_log_opt_spread_excluded": float(np.nanmedian(spread[elig & ~keep])) if (elig & ~keep).any() else np.nan}
            buckets = pd.Series(keep[elig]).groupby(tte_bucket(mat.tte_min[elig]).astype(str)).mean()
            rec.update({f"share_{k}": v for k, v in buckets.items()})
            sel.append(rec)
selection = pd.DataFrame(sel)
display(selection.round(4))

,underlying,horizon,model,eligible,retained,retained_share,weekend_share_retained,min_session_share,median_y_retained,median_y_excluded,median_log_opt_spread_retained,median_log_opt_spread_excluded,share_0-4h,share_16-24h,share_4-8h,share_8-16h
0,AVAX,30m,M1,2444,2444,1.0000,0.3106,1.0000,-10.2697,NaN,8.8273,NaN,1.0000,1.0000,1.0000,1.0000
1,AVAX,30m,M2a,2444,2223,0.9096,0.2982,0.7439,-10.2495,-10.5538,8.8049,9.3371,0.6771,0.9575,0.9727,0.9925
2,AVAX,30m,M2b,2444,2093,0.8564,0.3000,0.6220,-10.2488,-10.4754,8.8049,9.1660,0.6183,0.8452,0.9355,0.9762
3,AVAX,30m,M3,2444,1361,0.5569,0.2866,0.0000,-10.1156,-10.5651,8.8049,8.8793,0.3798,0.6241,0.6348,0.5782
4,AVAX,30m,M4,2444,863,0.3531,0.3024,0.0000,-10.0702,-10.4316,8.6657,8.9740,0.0367,0.5340,0.2656,0.4919
5,AVAX,60m,M1,2249,2249,1.0000,0.3206,1.0000,-9.4196,NaN,8.8285,NaN,1.0000,1.0000,1.0000,1.0000
6,AVAX,60m,M2a,2249,2090,0.9293,0.3081,0.7865,-9.4120,-9.6079,8.8061,9.3329,0.7419,0.9621,0.9723,0.9920
7,AVAX,60m,M2b,2249,1975,0.8782,0.3089,0.7286,-9.4132,-9.4928,8.8103,9.1359,0.6855,0.8542,0.9348,0.9748
8,AVAX,60m,M3,2249,1269,0.5643,0.3018,0.0000,-9.2971,-9.6386,8.8065,8.8772,0.4143,0.6174,0.6364,0.5703
9,AVAX,60m,M4,2249,804,0.3575,0.3209,0.0000,-9.2040,-9.5311,8.6797,8.9725,0.0434,0.5360,0.2688,0.4841


### Coefficients

HAR + clock coefficients of M1 (persistence structure), and the largest standardized added-block coefficients of M3 / M4.

In [9]:
har = []
top = []
for a, by_h in SCREEN.items():
    for h, r in by_h.items():
        m1 = r["fits"]["M1"]
        har.append({"underlying": a, "horizon": h, **dict(zip(m1.h_names, np.round(m1.phi, 4))), "smear": m1.smear})
        for m in ("M3", "M4"):
            fv = r["fits"][m]
            if fv.x_names:
                s = pd.Series(fv.gamma, index=fv.x_names)
                for name, v in s.reindex(s.abs().sort_values(ascending=False).index).head(10).items():
                    top.append({"underlying": a, "horizon": h, "model": m, "feature": name, "coef": v})
print("M1 HAR + clock coefficients:")
display(pd.DataFrame(har))
print("Largest added-block coefficients (standardized, after shrinkage):")
display(pd.DataFrame(top).round(4))

M1 HAR + clock coefficients:


,underlying,horizon,const,l5m,l30m,l1h,l4h,clock_fwd,clock_back,weekend,smear
0,AVAX,30m,-6.0259,0.0064,0.1663,0.0295,0.2672,0.4645,-0.0666,-0.0274,1.372338
1,AVAX,60m,-5.8126,0.0126,0.0594,0.0510,0.2882,0.4615,-0.0343,-0.0285,1.204546
2,BTC,30m,-6.1617,-0.0003,0.2174,0.0475,0.2944,1.5374,-0.9386,0.3468,1.386564
3,BTC,60m,-6.6094,0.0074,0.0755,0.1010,0.2668,1.3195,-0.7061,0.3061,1.230615
4,ETH,30m,-6.7015,0.0106,0.3396,-0.0223,0.1162,0.7007,-0.1458,0.0700,1.409878
5,ETH,60m,-7.5111,0.0165,0.2314,-0.0775,0.1300,0.7033,-0.0690,0.0496,1.271520
6,HYPE,30m,-8.2700,0.0074,0.1679,0.0157,0.0474,0.8433,-0.4595,-0.3544,1.338793
7,HYPE,60m,-8.7198,0.0067,0.0548,-0.0048,0.0673,0.6855,-0.3305,-0.4390,1.189984
8,SOL,30m,-6.6932,0.0128,0.1494,-0.0967,0.3967,0.8897,-0.2658,-0.0956,1.297007
9,SOL,60m,-6.4639,0.0128,0.0274,-0.0225,0.3929,0.7863,-0.1839,-0.0986,1.158902


Largest added-block coefficients (standardized, after shrinkage):


,underlying,horizon,model,feature,coef
0,AVAX,30m,M3,log_book_updates,0.0
1,AVAX,30m,M3,log_avg_trade,-0.0
2,AVAX,30m,M3,log_gk30,0.0
3,AVAX,30m,M3,log_range_bps,0.0
4,AVAX,30m,M3,log_park30,0.0
...,...,...,...,...,...
235,XRP,60m,M4,log_ask_depth20,-0.0
236,XRP,60m,M4,log_book_updates,0.0
237,XRP,60m,M4,skew025_d30,0.0
238,XRP,60m,M4,skew025_d60,0.0


### Physical forecast vs clock-allocated implied variance (descriptive)

$\log(\widehat W^{real}/\widetilde W^{imp})$ against $\log(RV/\widetilde W^{imp})$ on the evaluation rows. The median realized-to-implied log ratio is the in-sample variance-risk-premium proxy. This is not a P&L statement: a gap can be a rational premium (004 §10).

In [10]:
econ = []
for a, by_h in SCREEN.items():
    mat = INPUTS[a]["mat"]
    te = np.isin(mat.sessions, sorted(SCREEN_EVAL))
    for h, r in by_h.items():
        lW = mat.X[h]["log_W_imp_alloc"].to_numpy()
        real = np.log(mat.level[h]) - lW
        for m in ("M1", "M2b", "M3"):
            sc = SCORES[a][h][m]
            gap_hat = sc["yhat"].to_numpy() + np.log(r["fits"][m].smear) - lW
            ok = te & np.isfinite(gap_hat) & np.isfinite(real)
            econ.append({"underlying": a, "horizon": h, "model": m, "n": int(ok.sum()),
                         "corr_gap_hat_gap_real": float(np.corrcoef(gap_hat[ok], real[ok])[0, 1]) if ok.sum() > 10 else np.nan,
                         "median_log_rv_over_W": float(np.median(real[ok])) if ok.any() else np.nan})
display(pd.DataFrame(econ).round(4))

,underlying,horizon,model,n,corr_gap_hat_gap_real,median_log_rv_over_W
0,AVAX,30m,M1,1131,0.3744,0.0297
1,AVAX,30m,M2b,1049,0.4055,0.0297
2,AVAX,30m,M3,668,0.3191,0.1548
3,AVAX,60m,M1,1052,0.3974,0.2458
4,AVAX,60m,M2b,981,0.4072,0.2646
5,AVAX,60m,M3,620,0.3548,0.3734
6,BTC,30m,M1,1136,0.3718,-0.1132
7,BTC,30m,M2b,1085,0.3719,-0.1149
8,BTC,30m,M3,1077,0.3843,-0.1111
9,BTC,60m,M1,1059,0.3874,0.0629


## 5. Freeze the specification

All nuisance quantities, eligibility, penalties and coefficients are refitted on **all development sessions** (2026-09-22 → 10-07; CV folds inside them) and written to `notebooks/12_frozen_spec.json` with the code commit. Commit that file **before** scoring any confirmation session. An existing spec is never overwritten unless `NB12_REFREEZE=1`.

In [11]:
def git_commit():
    try:
        return subprocess.check_output(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


if MODE == "development" and FREEZE:
    if SPEC_PATH.exists() and not REFREEZE:
        print(f"{SPEC_PATH.name} exists: not overwritten (set NB12_REFREEZE=1 to replace it before confirmation).")
    else:
        spec = {"meta": {
            "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "git_commit": git_commit(), "vol_targets": vt.VOL_TARGETS_VERSION, "vol_har": vh.VOL_HAR_VERSION,
            "clock_baseline": CLOCK_BASELINE_VERSION, "full_information": fi.FULL_INFORMATION_VERSION,
            "development_sessions": DEV_SESSIONS, "confirmation_starts_after": DEV_END,
            "confirmation_min_sessions": CONFIRM_MIN_SESSIONS, "min_feature_coverage": MIN_FEATURE_COVERAGE,
            "lambda_grid": list(vh.LAMBDA_GRID), "en_grid": list(EN_GRID), "horizons": vt.HORIZONS,
            "primary_score": "QLIKE (Duan smearing)", "hierarchy": [list(p) for p in vh.HIERARCHY[1:]],
        }, "assets": {}}
        t0 = time.perf_counter()
        FROZEN_INPUTS = {}
        for a in ASSETS:
            b = prepare(a, set(DEV_SESSIONS))
            FROZEN_INPUTS[a] = b
            entry = {"nuisance": b["nuis"].to_dict(), "horizons": {}}
            for h in HORIZONS:
                r = fit_window(a, b, set(DEV_SESSIONS), h)
                entry["horizons"][h] = {
                    "specs": r["specs"], "lambda": r["lam"], "en_alpha": r["en_alpha"],
                    "models": {m: fv.to_dict() for m, fv in r["fits"].items()},
                }
            spec["assets"][a] = entry
            print(f"{a}: frozen ({time.perf_counter() - t0:.0f}s)")
        SPEC_PATH.write_text(json.dumps(spec, indent=1, default=float))
        print(f"Wrote {SPEC_PATH} ({SPEC_PATH.stat().st_size / 1e6:.2f} MB). Commit it before any confirmation scoring.")

AVAX: frozen (56s)
BTC: frozen (100s)
ETH: frozen (147s)
HYPE: frozen (193s)
SOL: frozen (241s)
XRP: frozen (282s)
Wrote D:\Projects\deribit_market_data\notebooks\12_frozen_spec.json (0.45 MB). Commit it before any confirmation scoring.


## 6. Confirmation (binding after 30 completed sessions)

Run with `NB12_MODE=confirmation`. The frozen nuisance, eligibility and coefficients are applied unchanged to every completed session after 2026-10-07. With fewer than 30 sessions the readout is a **non-binding checkpoint** and changes no model choice.

In [12]:
if MODE == "confirmation":
    spec = json.loads(SPEC_PATH.read_text())
    print(f"Frozen spec from commit {spec['meta']['git_commit'][:10]} created {spec['meta']['created_utc']}")
    conf_sessions = sorted(
        s for s in df["session_date"].unique()
        if s > spec["meta"]["confirmation_starts_after"]
        and pd.Timestamp(f"{s} 08:00", tz="UTC") <= df["state_time"].max()
    )
    binding = len(conf_sessions) >= spec["meta"]["confirmation_min_sessions"]
    print(f"Completed confirmation sessions: {len(conf_sessions)} -> "
          f"{'BINDING readout' if binding else 'NON-BINDING checkpoint'}")

    conf_rows, conf_metrics = [], []
    for a in ASSETS:
        entry = spec["assets"][a]
        g = asset_frame(a)
        panel = vt.build_panel(g)
        nuis = vh.Nuisance(**entry["nuisance"])
        raw, _ = raw_features(g, nuis)
        mat = vh.materialize(panel, nuis, raw)
        cr = np.isin(mat.sessions, conf_sessions)
        for h, hz in entry["horizons"].items():
            sc = {m: vh.row_scores(vh.FittedVol.from_dict(d), mat, h) for m, d in hz["models"].items()}
            nov = vh.non_overlapping(mat.tte_min, h)
            for m, s in sc.items():
                rows = cr & np.isfinite(s["qlike"].to_numpy())
                conf_metrics.append({"underlying": a, "horizon": h, "model": m, "n": int(rows.sum()),
                                     "qlike": float(s["qlike"][rows].mean()),
                                     "r2": vh.r2(mat.y[h][rows], s["yhat"].to_numpy()[rows])})
            for x, y_ in vh.HIERARCHY + [("M1c", "M1")]:
                for view, rows in (("all", cr), ("non_overlapping", cr & nov)):
                    c = vh.compare(sc[x], sc[y_], mat.sessions, rows, n_boot=N_BOOT)
                    conf_rows.append({"underlying": a, "horizon": h, "model": x, "vs": y_, "view": view, **c})
    confirmation = pd.DataFrame(conf_rows)
    display(pd.DataFrame(conf_metrics).round(4))
    display(confirmation[show].round(4))
else:
    print("Development mode: confirmation not run.")

Development mode: confirmation not run.


## 7. Interpretation gates

1. **Specification checks** (development): targets and inputs present for every asset; HAR coefficients positive and plausible; M1 beats M0; no model's QLIKE explodes. A failure here is fixed before freezing, not interpreted.
2. **Sensor hierarchy** (binding, confirmation only): M2a vs M1, M2b vs M2a, M3 vs M2b, M4 vs M3, judged by the session-paired QLIKE difference and its whole-session bootstrap interval, with the non-overlapping view required to agree in sign.
3. **Module 13 gate** (independent of 2): no alignment failures and finite, stable HAR forecasts. If M1 has an unconditional calibration bias, the training-only recalibrated **M1c** is the null that switching models must beat (004 §8).
4. **No economic claim.** A forecast–implied gap is a candidate only if it later predicts executable option P&L net of spread, fees and hedge costs.
5. **Terminal target** stays provisional until the settlement-average path simulation is built.